# Phase 4 — Model Training
XGBoost regressor, GroupKFold by plant_id, target = day

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.model import prepare_features, train_cv, evaluate_by_variety, save_model, FEATURE_COLS

df = prepare_features('../data/features.csv')
print(f'Shape: {df.shape}')
print(f'Features: {FEATURE_COLS}')

## Train — GroupKFold(5) by plant_id

In [ ]:
model, cv = train_cv(df, n_splits=5)
print(f"\nCV MAE:  {cv['mae_mean']:.3f} +/- {cv['mae_std']:.3f}")
print(f"CV RMSE: {cv['rmse_mean']:.3f} +/- {cv['rmse_std']:.3f}")
print(f"CV R2:   {cv['r2_mean']:.3f} +/- {cv['r2_std']:.3f}")

## Metrics แยก Variety

In [ ]:
metrics = evaluate_by_variety(df, cv['oof_pred'])
print(metrics.to_string(index=False))

## Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 5))
importance.plot.barh(ax=ax)
ax.set_title('Feature Importance (XGBoost)')
ax.set_xlabel('Score')
plt.tight_layout()
plt.savefig('../results/model_importance.png', dpi=80, bbox_inches='tight')
plt.show()

## Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, var in zip(axes, ['COS', 'GOK']):
    sub = df[df['variety'] == var]
    pred = cv['oof_pred'][sub.index]
    ax.scatter(sub['day'], pred, alpha=0.3, s=8)
    ax.plot([0,8],[0,8], 'r--', lw=1)
    ax.set_xlabel('Actual Day')
    ax.set_ylabel('Predicted Day')
    ax.set_title(var)
plt.suptitle('Predicted vs Actual (OOF)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/model_pred_vs_actual.png', dpi=80, bbox_inches='tight')
plt.show()

## Error by Day

In [ ]:
df2 = df.copy()
df2['pred'] = cv['oof_pred']
df2['abs_error'] = (df2['pred'] - df2['day']).abs()
err_by_day = df2.groupby(['variety','day'])['abs_error'].mean().unstack(0)
fig, ax = plt.subplots(figsize=(9, 4))
err_by_day.plot(marker='o', ax=ax)
ax.set_xlabel('Day')
ax.set_ylabel('MAE')
ax.set_title('MAE by Day')
plt.tight_layout()
plt.savefig('../results/model_error_by_day.png', dpi=80, bbox_inches='tight')
plt.show()

## Save Model

In [ ]:
save_model(model, '../models/xgb_model.json')
print('Done')

## สรุป Phase 4

In [ ]:
model_notes = {
    'cv_mae':  round(cv['mae_mean'], 3),
    'cv_rmse': round(cv['rmse_mean'], 3),
    'cv_r2':   round(cv['r2_mean'], 3),
    'top_features': ['a_mean (~46%)', 'pct_yellow (~24%)', 'pct_brown (~8%)'],
    'weak_days': ['D0', 'D7', 'D8'],
    'decision': 'Regression + map to grade (Phase 5) — ไม่ train classifier ตรง',
    'notes': (
        'a_mean ครองความสำคัญเด่นมาก (~46%) รองมาคือ pct_yellow (~24%) และ pct_brown (~8%) '
        '- รวม 3 ตัวนี้ ~78% ของ importance | '
        'error สูงที่ปลายช่วง (D0 และ D7-D8) เป็น boundary effect '
        '(regression เอนค่าสุดขอบเข้าหากลาง) | '
        'per-variety ใกล้กัน (COS R2=0.888, GOK R2=0.898) ไม่ต้องแยก model | '
        'variety_enc importance ต่ำสุด ตอกย้ำพันธุ์ส่งผลน้อยเทียบกับ feature สี | '
        'area_ratio/pct_green importance ต่ำเพราะ collinearity กับ a_mean/pct_yellow '
        'ไม่ใช่ว่าไม่มีประโยชน์'
    ),
}
for k, v in model_notes.items():
    print(f'{k}: {v}')